# L2 Escape Model

Consequent Reversibility was not included in the final escape models because, in this subgroup structure, it is partially encoded by Goal_Type and produced unstable estimates when entered together with Goal_Type.

Import Libraries

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Overall Subgroup:
1. Create escape_L2
2. Counts and Proportions
3. Logistic Models
4. Model Comparison

Read data

In [41]:
subgroup_theoretical = pd.read_csv("../../data/processed/subgroup_theoretical.csv")

1. Create escape_2

In [42]:
subgroup_theoretical["escape_L2"] = (
    subgroup_theoretical["Response_Full"] != "L2_other"
).astype(int)

2. Counts and Proportions

In [43]:
escape_L2_counts = pd.crosstab(
    subgroup_theoretical["escape_L2"],
    subgroup_theoretical["Response_Full"])

escape_L2_counts

Response_Full,L1_transfer,L2_other,correct
escape_L2,,,
0,0,38,0
1,96,0,61


In [44]:
escape_L2_props = subgroup_theoretical["escape_L2"].value_counts(normalize = True)
escape_L2_props

escape_L2
1    0.808081
0    0.191919
Name: proportion, dtype: float64

Counts and Proportions by Condition

In [45]:
counts_condition = pd.crosstab(
    [subgroup_theoretical["Goal_Type"], subgroup_theoretical["Agent"], subgroup_theoretical["Consequent_Reversibility"]],
    subgroup_theoretical["escape_L2"],
    margins = True)

counts_condition

escape_L2                                          0    1  All
Goal_Type         Agent Consequent_Reversibility              
goal_frequent     0     Yes                       14   28   42
                  1     Yes                        2   22   24
goal_non_frequent 0     No                         9   33   42
                  1     No                         1   23   24
no_goal           0     Yes                        8   34   42
                  1     Yes                        4   20   24
All                                               38  160  198

In [46]:
props_condition = pd.crosstab(
    [subgroup_theoretical["Goal_Type"], subgroup_theoretical["Agent"], subgroup_theoretical["Consequent_Reversibility"]],
    subgroup_theoretical["escape_L2"],
    normalize= "index")

props_condition

escape_L2                                                0         1
Goal_Type         Agent Consequent_Reversibility                    
goal_frequent     0     Yes                       0.333333  0.666667
                  1     Yes                       0.083333  0.916667
goal_non_frequent 0     No                        0.214286  0.785714
                  1     No                        0.041667  0.958333
no_goal           0     Yes                       0.190476  0.809524
                  1     Yes                       0.166667  0.833333

Agency appears to help participants escape L2. 

The condition with the strongest L1 transfer (goal_non_frequent, agent=1) also exhibits the highest L2 escape rate (95.8%). This suggests that failure does not occur at the L2 suppression stage, but at the subsequent resolution stage.

In addition, if there is a goal representation available, agency seems to help participants commit by leaving the L2 space; however, if there is no goal representation, agency seems to have much less to work with. 

3. Logistic Models

Import Libraries

In [47]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2



Binary Outcome

In [48]:
subgroup_theoretical["escape_L2"] = (
    subgroup_theoretical["Response_Full"] != "L2_other"
).astype(int)

In [49]:
subgroup_theoretical["escape_L2"].value_counts()

escape_L2
1    160
0     38
Name: count, dtype: int64

**Model_1: escape L2**
escape_L2 ~ Agent x Goal_Type

In [50]:
model_1 = smf.gee("escape_L2 ~ C(Agent) * C(Goal_Type)", 
                  groups="Participant_ID", 
                  data= subgroup_theoretical, 
                  family=sm.families.Binomial())
results_1 = model_1.fit()
print(results_1.summary())


                               GEE Regression Results                              
Dep. Variable:                   escape_L2   No. Observations:                  198
Model:                                 GEE   No. clusters:                       33
Method:                        Generalized   Min. cluster size:                   6
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 6.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 16 Jun 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         18:04:15
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
Intercept   

**Model_2: resolve L1**
resolve_L1 ~ Agent x Goal_Type

In [51]:
df_conflict = subgroup_theoretical[subgroup_theoretical['Response_Full'].isin(['correct', 'L1_transfer'])].copy()

In [52]:
df_conflict['resolve_L1'] = (df_conflict['Response_Full'] == 'correct').astype(int)

In [53]:
# interactive

model_2 = smf.gee("resolve_L1 ~ C(Agent) * C(Goal_Type)", 
                  groups="Participant_ID", 
                  data= df_conflict, 
                  family=sm.families.Binomial())
results_2 = model_2.fit()
print(results_2.summary())

                               GEE Regression Results                              
Dep. Variable:                  resolve_L1   No. Observations:                  157
Model:                                 GEE   No. clusters:                       33
Method:                        Generalized   Min. cluster size:                   1
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 4.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 16 Jun 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         18:04:15
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
Intercept   

In [54]:
# additive
model_2_2 = smf.gee("resolve_L1 ~ C(Agent) + C(Goal_Type)", 
                  groups="Participant_ID", 
                  data= df_conflict, 
                  family=sm.families.Binomial())
results_2_2 = model_2_2.fit()
print(results_2_2.summary())

                               GEE Regression Results                              
Dep. Variable:                  resolve_L1   No. Observations:                  157
Model:                                 GEE   No. clusters:                       33
Method:                        Generalized   Min. cluster size:                   1
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 4.8
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 16 Jun 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         18:04:16
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -0.

## Actors Subgroup:
1. Create escape_L2
2. Counts and Proportions
3. Logistic Models
4. Model Comparison

Read data

In [55]:
subgroup_actors = subgroup_theoretical[subgroup_theoretical["Focus"] == "I"].copy()

1. Create escape_L2

In [56]:
subgroup_actors["escape_L2"] = (
    subgroup_actors["Response_Full"] != "L2_other"
).astype(int)

2. Counts and Proportions

In [57]:
escape_L2_counts = pd.crosstab(
    subgroup_actors["escape_L2"],
    subgroup_actors["Response_Full"])

escape_L2_counts

Response_Full,L1_transfer,L2_other,correct
escape_L2,,,
0,0,24,0
1,53,0,40


In [58]:
escape_L2_props = subgroup_actors["escape_L2"].value_counts(normalize = True)
escape_L2_props

escape_L2
1    0.8
0    0.2
Name: proportion, dtype: float64

Counts and Proportions by Condition

In [59]:
counts_condition = pd.crosstab(
    [subgroup_actors["Goal_Type"], subgroup_actors["Agent"], subgroup_actors["Consequent_Reversibility"]],
    subgroup_actors["escape_L2"],
    margins = True)

counts_condition

escape_L2                                          0   1  All
Goal_Type         Agent Consequent_Reversibility             
goal_frequent     0     Yes                        8  16   24
                  1     Yes                        2  14   16
goal_non_frequent 0     No                         8  16   24
                  1     No                         1  15   16
no_goal           0     Yes                        3  21   24
                  1     Yes                        2  14   16
All                                               24  96  120

In [60]:
props_condition = pd.crosstab(
    [subgroup_actors["Goal_Type"], subgroup_actors["Agent"], subgroup_actors["Consequent_Reversibility"]],
    subgroup_actors["escape_L2"],
    normalize= "index")

props_condition

escape_L2                                                0         1
Goal_Type         Agent Consequent_Reversibility                    
goal_frequent     0     Yes                       0.333333  0.666667
                  1     Yes                       0.125000  0.875000
goal_non_frequent 0     No                        0.333333  0.666667
                  1     No                        0.062500  0.937500
no_goal           0     Yes                       0.125000  0.875000
                  1     Yes                       0.125000  0.875000

3. Logistic Models

Binary Outcome

In [61]:
subgroup_actors["escape_L2"] = (
    subgroup_actors["Response_Full"] != "L2_other"
).astype(int)

In [62]:
subgroup_actors["escape_L2"].value_counts()

escape_L2
1    96
0    24
Name: count, dtype: int64

4. Model comparisons

**Model 1: Escape L2**: 
escape_L2 ~ Agent x Goal_Type

In [63]:
model_1 = smf.gee("escape_L2 ~ C(Agent) * C(Goal_Type)", 
                  groups="Participant_ID", 
                  data= subgroup_actors, 
                  family=sm.families.Binomial())
results_1 = model_1.fit()
print(results_1.summary())

                               GEE Regression Results                              
Dep. Variable:                   escape_L2   No. Observations:                  120
Model:                                 GEE   No. clusters:                       20
Method:                        Generalized   Min. cluster size:                   6
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 6.0
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 16 Jun 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         18:04:16
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
Intercept   

**Model_2: resolve L1**
resolve_L1 ~ Agent x Goal_Type

In [64]:
df_conflict_a = subgroup_actors[subgroup_actors['Response_Full'].isin(['correct', 'L1_transfer'])].copy()

In [65]:
df_conflict_a['resolve_L1'] = (df_conflict_a['Response_Full'] == 'correct').astype(int)

In [66]:
model_2 = smf.gee("resolve_L1 ~ C(Agent) * C(Goal_Type)", 
                  groups="Participant_ID", 
                  data= df_conflict_a, 
                  family=sm.families.Binomial())
results_2 = model_2.fit()
print(results_2.summary())

                               GEE Regression Results                              
Dep. Variable:                  resolve_L1   No. Observations:                   93
Model:                                 GEE   No. clusters:                       20
Method:                        Generalized   Min. cluster size:                   1
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 4.7
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 16 Jun 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         18:04:16
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
Intercept   

In [67]:
# additive
model_2_2 = smf.gee("resolve_L1 ~ C(Agent) + C(Goal_Type)", 
                  groups="Participant_ID", 
                  data= df_conflict_a, 
                  family=sm.families.Binomial())
results_2_2 = model_2_2.fit()
print(results_2_2.summary())

                               GEE Regression Results                              
Dep. Variable:                  resolve_L1   No. Observations:                   93
Model:                                 GEE   No. clusters:                       20
Method:                        Generalized   Min. cluster size:                   1
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 4.7
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Tue, 16 Jun 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         18:04:16
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                             0.

# Short Comparison:
1. Does the same predictor structure hold?
2. Does effect size/ model fit improve?